<a href="https://colab.research.google.com/github/OmegaTrees/google-colab-telegram-bots/blob/main/Multi_Purpose_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# @title 🌑  Multi Purpose Tool v6.0
# @markdown **Tools:** Video Editor | Smart Drive Downloader | Merger | Planner | YouTube Uploader (Custom Auth)

import os
import sys
import subprocess
import time
import json
import shutil
import re
import urllib.parse
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import drive, files

# ==========================================
# 0. SYSTEM CONFIG & AUTH
# ==========================================

# USER PROVIDED CLIENT CONFIG
CLIENT_CONFIG = {
    "installed": {
        "client_id": "",
        "client_secret": "",
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "auth_provider_x509_cert_url": "https://www.googleapis.com/oauth2/v1/certs",
        "redirect_uris": ["http://localhost"]
    }
}

def install_deps():
    if not os.path.exists("/usr/bin/aria2c"):
        subprocess.run('apt-get update -qq && apt-get install -y aria2 ffmpeg -qq > /dev/null', shell=True)
    try:
        import google_auth_oauthlib
    except ImportError:
        subprocess.run('pip install google-auth-oauthlib google-api-python-client > /dev/null', shell=True)

install_deps()

from google.auth.transport.requests import Request
from googleapiclient.http import MediaFileUpload
import google_auth_oauthlib.flow
import googleapiclient.discovery

# ==========================================
# 1. UI STYLING
# ==========================================
style = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.3/font/bootstrap-icons.min.css">
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;500;700&display=swap" rel="stylesheet">
<link href="https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;700&display=swap" rel="stylesheet">
<style>
    .jupyter-widgets { font-family: 'Roboto', sans-serif; background: #0d1117; color: #c9d1d9; }
    .app-container { max-width: 950px; margin: 0 auto; padding-bottom: 50px; }

    /* TABS */
    .custom-tab .widget-toggle-button {
        background: #161b22 !important; color: #8b949e !important;
        border: 1px solid #30363d !important; border-bottom: none !important;
        margin-right: 5px; border-radius: 8px 8px 0 0;
    }
    .custom-tab .widget-toggle-button.mod-active {
        background: #1f6feb !important; color: white !important; font-weight: bold;
    }

    /* CARDS */
    .mui-card {
        background: #161b22; border: 1px solid #30363d;
        border-radius: 0 12px 12px 12px; padding: 24px;
        box-shadow: 0 10px 30px rgba(0,0,0,0.5); margin-bottom: 20px;
    }
    .card-header {
        font-size: 18px; font-weight: 700; color: #ffffff;
        margin-bottom: 20px; display: flex; align-items: center;
        border-bottom: 1px solid #30363d; padding-bottom: 12px;
        font-family: 'JetBrains Mono', monospace;
    }
    .icon-header { margin-right: 12px; color: #1f6feb; font-size: 22px; }

    /* INPUTS */
    .widget-text input, .widget-textarea textarea {
        background: #0d1117 !important; color: #58a6ff !important;
        border: 1px solid #30363d !important; border-radius: 6px !important;
        padding: 10px !important; font-family: 'JetBrains Mono', monospace;
    }
    .widget-dropdown select { background: #0d1117 !important; color: #fff !important; border: 1px solid #30363d !important; }

    .track-row { display: flex; align-items: center; background: #0d1117; border: 1px solid #30363d; border-radius: 6px; padding: 8px; margin-bottom: 8px; }
    .badge { padding: 3px 6px; border-radius: 4px; font-size: 10px; font-weight: 700; color: white; margin-right: 8px;}
    .badge-vid { background: #79c0ff; color: #000; }
    .badge-aud { background: #7ee787; color: #000; }
    .log-box { background: #000; color: #1f6feb; font-family: 'JetBrains Mono', monospace; padding: 15px; border-radius: 8px; height: 200px; overflow-y: auto; font-size: 12px; border: 1px solid #30363d; }
</style>
"""
display(HTML(style))

state = {'video_file': 'input_video.mkv', 'audio_file': 'input_audio.m4a', 'has_external_audio': False, 'tracks': [], 'raw_streams': [], 'creds': None}
log_output = widgets.Output()
log_output.add_class("log-box")
def log(msg):
    with log_output: print(f">> {msg}")

# ==========================================
# MODULE 1: VIDEO EDITOR
# ==========================================
ve_head = widgets.HTML('<div class="card-header"><i class="bi bi-film icon-header"></i> VIDEO EDITOR</div>')
ve_vid = widgets.Text(placeholder='Video Link', layout=widgets.Layout(width='99%'))
ve_aud = widgets.Text(placeholder='Ext Audio Link (Opt)', layout=widgets.Layout(width='99%'))
ve_btn_an = widgets.Button(description='Analyze', button_style='primary', icon='search', layout=widgets.Layout(width='100%'))
ve_preset = widgets.Dropdown(options=[('Original (Copy)', 'copy'), ('BD/IMAX Original (CRF 17)', 'original_bd'), ('480p->1080p (Lanczos)', 'scale=1920:1080:flags=lanczos'), ('Bluray Sharpen (CAS)', 'cas=0.4')], value='copy', description='Preset:', layout=widgets.Layout(width='98%'))
ve_audio_mode = widgets.ToggleButtons(options=['Multi', 'Single'], button_style='info', layout=widgets.Layout(width='150px'))
ve_tracks = widgets.VBox([])
ve_sample = widgets.Dropdown(options=[('15s', '15'), ('30s', '30')], value='30', description='Sample:', layout=widgets.Layout(width='140px'))
ve_out = widgets.Text(value='output.mkv', placeholder='Name', layout=widgets.Layout(width='99%'))
ve_dest = widgets.ToggleButtons(options=['Local', 'GDrive'], button_style='success')
ve_btn_sam = widgets.Button(description='Sample', button_style='info', layout=widgets.Layout(width='48%'))
ve_btn_run = widgets.Button(description='Encode', button_style='success', layout=widgets.Layout(width='48%'))

tab_video = widgets.VBox([ve_head, ve_vid, ve_aud, ve_btn_an, widgets.HTML("<hr style='border-color:#30363d'>"), ve_preset, widgets.HBox([widgets.Label("Audio Mode:"), ve_audio_mode]), ve_tracks, widgets.HTML("<hr style='border-color:#30363d'>"), widgets.HBox([ve_out, ve_sample]), ve_dest, widgets.HBox([ve_btn_sam, ve_btn_run])])
tab_video.add_class("mui-card")

def render_tracks(change=None):
    ve_tracks.children = []
    state['tracks'] = []
    if not state['raw_streams']: return
    if ve_audio_mode.value == 'Single':
        opts = [(f"ID {s['index']} | {s['lang']}", i) for i, s in enumerate(state['raw_streams']) if s['type'] == 'audio']
        ve_tracks.children = [widgets.Dropdown(options=opts, description='Select:', layout=widgets.Layout(width='98%'))]
        state['single_widget'] = ve_tracks.children[0]
    else:
        rows = []
        uid, pos = 0, 1
        for s in state['raw_streams']:
            if s['type'] == 'audio':
                badge = "badge-aud"
                chk = widgets.Checkbox(value=True, layout=widgets.Layout(width='30px'))
                info = widgets.HTML(f"<div style='flex-grow:1; margin-left:10px; color:#c9d1d9; font-size:12px'><span class='badge {badge}'>AUDIO</span> ID {s['index']} | {s['lang'].upper()}</div>")
                txt = widgets.Text(value=s['lang'].upper(), layout=widgets.Layout(width='80px'))
                dd = widgets.Dropdown(options=[(str(i), i) for i in range(1, 11)], value=pos, layout=widgets.Layout(width='50px'))
                row = widgets.HBox([chk, info, txt, dd])
                row.add_class("track-row")
                state['tracks'].append({'idx': s['index'], 'chk': chk, 'name': txt, 'pos': dd})
                rows.append(row)
                pos += 1
        ve_tracks.children = tuple(rows)
ve_audio_mode.observe(render_tracks, names='value')

def ve_analyze(b):
    log_output.clear_output()
    state['raw_streams'] = []
    vid = ve_vid.value.strip()
    if not vid: return log("Error: No Video Link")
    ve_btn_an.disabled = True; ve_btn_an.description = "Fetching..."
    if os.path.exists(state['video_file']): os.remove(state['video_file'])
    subprocess.run(f'aria2c -x 16 -s 16 -k 1M -o "{state["video_file"]}" "{vid}"', shell=True)
    aud = ve_aud.value.strip()
    if aud:
        state['has_external_audio'] = True
        if os.path.exists(state['audio_file']): os.remove(state['audio_file'])
        subprocess.run(f'aria2c -x 16 -s 16 -k 1M -o "{state["audio_file"]}" "{aud}"', shell=True)
    try:
        res = subprocess.check_output(f'ffprobe -v error -show_entries stream=index,codec_type,codec_name:stream_tags=language -of json "{state["video_file"]}"', shell=True)
        for s in json.loads(res)['streams']:
            stype = 'video' if s['codec_type'] == 'video' else 'audio'
            state['raw_streams'].append({'type': stype, 'file_index': 0, 'index': s['index'], 'lang': s.get('tags', {}).get('language', 'UNK'), 'codec': s.get('codec_name', '')})
        if state['has_external_audio']:
             res_a = subprocess.check_output(f'ffprobe -v error -show_entries stream=index,codec_type,codec_name:stream_tags=language -of json "{state["audio_file"]}"', shell=True)
             for s in json.loads(res_a)['streams']:
                 state['raw_streams'].append({'type': 'audio', 'file_index': 1, 'index': s['index'], 'lang': s.get('tags', {}).get('language', 'UNK'), 'codec': s.get('codec_name', '')})
        render_tracks()
        ve_btn_run.disabled = False
        ve_btn_sam.disabled = False
        log(f"Analyzed {len(state['raw_streams'])} streams.")
    except Exception as e: log(f"Error: {e}")
    ve_btn_an.disabled = False; ve_btn_an.description = "Analyze"
ve_btn_an.on_click(ve_analyze)

def ve_execute(b):
    is_sam = (b.description == 'Sample')
    log_output.clear_output()
    cmd_in = f'-i "{state["video_file"]}" '
    if state['has_external_audio']: cmd_in += f'-i "{state["audio_file"]}" '
    maps = "-map 0:v "
    metas = ""
    if ve_audio_mode.value == 'Single':
        idx = state['single_widget'].value
        s = [x for x in state['raw_streams'] if x['type'] == 'audio'][idx]
        maps = f"-map 0:v -map {s['file_index']}:{s['index']} "
        metas = f'-metadata:s:a:0 title="{s["lang"].upper()}" '
    else:
        kept = [t for t in state['tracks'] if t['chk'].value]
        kept.sort(key=lambda x: x['pos'].value)
        for i, t in enumerate(kept):
            maps += f"-map 0:{t['idx']} " # Simplified map logic
            metas += f'-metadata:s:a:{i} title="{t["name"].value}" '
    f_val = ve_preset.value
    if f_val == 'copy': v_codec = "-c:v copy"
    elif f_val == 'original_bd': v_codec = "-c:v libx264 -crf 17 -preset slow"
    else: v_codec = f'-vf "{f_val}" -c:v libx264 -crf 17 -preset fast'
    out = f"sample_{ve_sample.value}s.mkv" if is_sam else ve_out.value
    t_flag = f"-t {ve_sample.value}" if is_sam else ""
    cmd = f'ffmpeg -y {cmd_in} {t_flag} {maps} {v_codec} -c:a copy {metas} -disposition:a:0 default "{out}"'
    log(f"🚀 Executing ({f_val})...")
    subprocess.run(cmd, shell=True)
    if os.path.exists(out):
        log("✅ Done!")
        if ve_dest.value == 'GDrive':
            if not os.path.exists('/content/drive'): drive.mount('/content/drive')
            shutil.copy(out, f"/content/drive/MyDrive/Colab_Downloads/{out}")
            log("Saved to Drive.")
        else: files.download(out)
    else: log("Error.")
ve_btn_sam.on_click(ve_execute); ve_btn_run.on_click(ve_execute)

# ==========================================
# MODULE 2: SMART DOWNLOADER (DRIVE)
# ==========================================
dl_head = widgets.HTML('<div class="card-header"><i class="bi bi-cloud-arrow-down icon-header"></i> SMART DRIVE DOWNLOADER</div>')
dl_url = widgets.Text(placeholder='Direct Link', layout=widgets.Layout(width='99%'))
dl_name = widgets.Text(placeholder='Filename (Auto)', layout=widgets.Layout(width='49%'))
dl_path = widgets.Text(value='Downloads', placeholder='Drive Folder', layout=widgets.Layout(width='49%'))
dl_btn = widgets.Button(description='Download to Drive', button_style='success', icon='google-drive', layout=widgets.Layout(width='100%'))
tab_down = widgets.VBox([dl_head, dl_url, widgets.HBox([dl_name, dl_path]), widgets.HTML("<br>"), dl_btn])
tab_down.add_class("mui-card")

def dl_run(b):
    url = dl_url.value.strip()
    if not url: return log("No URL")
    fname = dl_name.value.strip() or "auto"
    if fname == "auto": fname = urllib.parse.unquote(url.split('/')[-1].split('?')[0]) or "download.dat"
    fname = re.sub(r'[\\/*?:"<>|]', "", fname)
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    folder = f"/content/drive/MyDrive/{dl_path.value.strip()}"
    if not os.path.exists(folder): os.makedirs(folder)
    log(f"⬇️ Downloading to Drive: {fname}")
    try:
        parsed = urllib.parse.urlparse(url)
        domain = f'{parsed.scheme}://{parsed.netloc}/'
        ua = "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        cmd = ['aria2c', '-x', '16', '-s', '16', '-k', '1M', f'--user-agent={ua}', f'--header=Referer: {domain}', f'--header=Origin: {domain[:-1]}', '-d', folder, '-o', fname, url]
        subprocess.run(cmd)
        if os.path.exists(f"{folder}/{fname}"): log("✅ Saved to Google Drive.")
        else: log("❌ Download Failed.")
    except Exception as e: log(f"Error: {e}")
dl_btn.on_click(dl_run)

# ==========================================
# MODULE 3: MERGER, PLANNER, UPLOADER
# ==========================================
# Merger
mg_head = widgets.HTML('<div class="card-header"><i class="bi bi-intersect icon-header"></i> LOSSLESS MERGER</div>')
mg_u1 = widgets.Text(placeholder='URL 1', layout=widgets.Layout(width='99%'))
mg_u2 = widgets.Text(placeholder='URL 2', layout=widgets.Layout(width='99%'))
mg_cnt = widgets.Dropdown(options=['mp4', 'mkv'], value='mp4', description='Container:')
mg_btn = widgets.Button(description='Merge', button_style='warning', layout=widgets.Layout(width='100%'))
tab_merge = widgets.VBox([mg_head, mg_u1, mg_u2, mg_cnt, mg_btn])
tab_merge.add_class("mui-card")
def mg_run(b):
    urls = [u for u in [mg_u1.value, mg_u2.value] if u.strip()]
    if len(urls) < 2: return log("Need 2 URLs")
    log("🔗 Merging...")
    subprocess.run("rm -f /content/m_*.mp4 /content/list.txt", shell=True)
    files = []
    for i, u in enumerate(urls):
        fname = f"m_{i}.{mg_cnt.value}"
        subprocess.run(f'aria2c -x 16 -o "{fname}" "{u}"', shell=True)
        files.append(fname)
    with open("list.txt", "w") as f:
        for n in files: f.write(f"file '/content/{n}'\n")
    subprocess.run(f'ffmpeg -y -f concat -safe 0 -i list.txt -c copy -map 0 "/content/merged.{mg_cnt.value}"', shell=True)
    if os.path.exists(f"/content/merged.{mg_cnt.value}"): files.download(f"/content/merged.{mg_cnt.value}")
mg_btn.on_click(mg_run)

# Planner
pl_head = widgets.HTML('<div class="card-header"><i class="bi bi-pencil-square icon-header"></i> SCRIPT GEN</div>')
pl_topic = widgets.Text(placeholder='Topic', layout=widgets.Layout(width='48%'))
pl_aud = widgets.Text(placeholder='Audience', layout=widgets.Layout(width='48%'))
pl_btn = widgets.Button(description='Gen Prompt', button_style='info')
pl_out = widgets.Textarea(layout=widgets.Layout(width='99%', height='80px'))
tab_plan = widgets.VBox([pl_head, widgets.HBox([pl_topic, pl_aud]), pl_btn, pl_out])
tab_plan.add_class("mui-card")
def pl_run(b): pl_out.value = f"Act as educator. Create script on {pl_topic.value} for {pl_aud.value}. Original, neutral tone."
pl_btn.on_click(pl_run)

# Uploader
up_head = widgets.HTML('<div class="card-header"><i class="bi bi-youtube icon-header"></i> YOUTUBE UPLOADER</div>')
up_code = widgets.Text(placeholder='Paste Auth Code Here', layout=widgets.Layout(width='70%'))
up_auth = widgets.Button(description='Get Link', button_style='danger')
up_file = widgets.Text(placeholder='Filename or Link', layout=widgets.Layout(width='99%'))
up_title = widgets.Text(placeholder='Video Title', layout=widgets.Layout(width='99%'))
up_btn = widgets.Button(description='Start Upload', button_style='success', layout=widgets.Layout(width='100%'))
tab_up = widgets.VBox([up_head, widgets.HBox([up_code, up_auth]), up_file, up_title, up_btn])
tab_up.add_class("mui-card")

def up_auth_run(b):
    flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_config(CLIENT_CONFIG, ["https://www.googleapis.com/auth/youtube.upload"])
    flow.redirect_uri = "http://localhost"
    url, _ = flow.authorization_url(prompt='consent')
    log(f"🔗 AUTH LINK: {url}")

def up_run(b):
    code = up_code.value.strip()
    if "code=" in code: code = code.split("code=")[1].split("&")[0]
    if "%" in code: code = urllib.parse.unquote(code)
    if not state.get('creds') and code:
        try:
            flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_config(CLIENT_CONFIG, ["https://www.googleapis.com/auth/youtube.upload"])
            flow.redirect_uri = "http://localhost"
            flow.fetch_token(code=code)
            state['creds'] = flow.credentials
            log("✅ Auth Success")
        except: return log("Auth Failed")
    src = up_file.value.strip()
    if src.startswith("http"):
        log("⬇️ Mirroring...")
        subprocess.run(f'aria2c -x 16 -o "upload.mp4" "{src}"', shell=True)
        src = "upload.mp4"
    if not os.path.exists(src): return log("File not found")
    log("🚀 Uploading...")
    youtube = googleapiclient.discovery.build("youtube", "v3", credentials=state['creds'])
    media = MediaFileUpload(src, chunksize=64*1024*1024, resumable=True)
    req = youtube.videos().insert(part="snippet,status", body={"snippet": {"categoryId": "22", "title": up_title.value}, "status": {"privacyStatus": "private"}}, media_body=media)
    resp = None
    while resp is None:
        status, resp = req.next_chunk()
        if status: log(f"⬆️ {int(status.progress()*100)}%")
    log(f"✅ Uploaded: {resp['id']}")
up_auth.on_click(up_auth_run); up_btn.on_click(up_run)

# MAIN
tabs = widgets.Tab(children=[tab_video, tab_down, tab_merge, tab_plan, tab_up])
tabs.set_title(0, "Editor"); tabs.set_title(1, "Downloader"); tabs.set_title(2, "Merger"); tabs.set_title(3, "Planner"); tabs.set_title(4, "Uploader")
tabs.add_class("custom-tab")
ui = widgets.VBox([tabs, log_output])
ui.add_class("app-container")
clear_output(); display(ui)